In [1]:
from sentimentLDA import *
from gensim.models import Word2Vec
import os
import urllib
import tarfile
import dill
import pandas as pd
from tqdm import tqdm
from nltk.corpus import sentiwordnet as swn
import numpy as np
tqdm.pandas()
vocabSize = 50000

C:\Libraries\anaconda3\lib\site-packages\tqdm\std.py:697: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [2]:
word_vectors = Word2Vec.load("./sentiment140-word2vec.model").wv

In [3]:
dataframe = pd.read_csv("./sentiment140-cleaned.csv", encoding="ISO-8859-1", engine="python")

In [4]:
n=1600
df = dataframe.sample(n=n, random_state=231)
df = df.reset_index(drop=True)

In [5]:
cluster_good = ["good", "nice", "cool", "lovely", "wonderful", "great", "awesome", "fantastic", "amazing", "fun", "excellent"]
cluster_bad = ["bad", "horrible", "terrible", "awful", "worst", "shitty", "crappy", "sucks", "hate"]
def sentiFun(word):
    if word in word_vectors.vocab:
        posScore = 1 - np.average(word_vectors.distances(word, cluster_good))
        negScore = 1 - np.average(word_vectors.distances(word, cluster_bad))
        return (posScore, negScore)
    return (0, 0)

In [6]:
def getTweetSentiment(tweetIndex, words):
    probabilities = []
    for wordIndex in range(len(words)):
        probabilities.append(sampler.conditionalDistribution(tweetIndex, wordIndex))
    probabilities = np.mean(probabilities, axis=0)
    topic = 0
    topicProb = 0
    for i in range(len(probabilities)):
        sentiments = probabilities[i]
        probability = sentiments[0] + sentiments[1]
        if probability > topicProb:
            topicProb = probability
            topic = i
    (_, _, topicClusterGood) = topicSentimentList[i*2 + 1]
    (_, _, topicClusterBad)  = topicSentimentList[i*2]
    topicClusterGood = [x for x in topicClusterGood if x in word_vectors.vocab]
    topicClusterBad  = [x for x in topicClusterBad if x in word_vectors.vocab]
    wordvectors = []
    for word in words:
        if not (word in word_vectors.vocab):
            continue
        wordvectors.append(word_vectors[word])
    if len(wordvectors) == 0:
        return 0
    sentencevector = np.mean(wordvectors, axis=0)

    positive = 1-np.average(word_vectors.distances(sentencevector, cluster_good))
    negative = 1-np.average(word_vectors.distances(sentencevector, cluster_bad))
    topicPositive = 1-np.average(word_vectors.distances(sentencevector, topicClusterGood))
    topicNegative = 1-np.average(word_vectors.distances(sentencevector, topicClusterBad))
    return positive - negative + (topicPositive - topicNegative)

In [10]:
seeds = [100, 231, 123, 1, 2]
alphas = [0.1, 0.3, 0.5, 0.7, 1, 1.5, 2, 2.5]
topicSentimentList = []
results = []
for alpha in alphas:
    res = []
    for seed in seeds:
        sampler = SentimentLDAGibbsSampler(10, alpha, 0.1, 0.3, seed=seed)
        sampler.run(tqdm(df['text'].tolist()), sentiFun, 100, "./lda.dll", True)
        topicSentimentList = sampler.getTopKWords(25)
        df['predicted'] = df.progress_apply(lambda row: getTweetSentiment(row.name, row['text'].split()), axis=1)
        df['predicted'] = df.progress_apply(lambda row: 4 if row['predicted'] > 0 else 0, axis=1)
        df['predict_correct'] = df.progress_apply(lambda row: row['sentiment'] == row['predicted'], axis=1)
        accuracy = df['predict_correct'].value_counts()[True] / n
        res.append(accuracy)
    results.append(res)

100%|██████████| 1600/1600 [00:00<00:00, 97802.10it/s]


In [11]:
print(results)

[[0.626875, 0.65375, 0.679375, 0.681875, 0.659375], [0.671875, 0.60375, 0.66625, 0.6825, 0.671875], [0.673125, 0.685625, 0.665625, 0.610625, 0.676875], [0.675625, 0.67, 0.68875, 0.67125, 0.67625], [0.605625, 0.651875, 0.67875, 0.671875, 0.683125], [0.681875, 0.584375, 0.635, 0.634375, 0.688125], [0.5775, 0.658125, 0.675, 0.600625, 0.665625], [0.665625, 0.688125, 0.664375, 0.6125, 0.6775]]


In [15]:
for res in results:
    print(res)

[0.666875, 0.66875, 0.65125, 0.684375, 0.680625]
[0.613125, 0.674375, 0.694375, 0.696875, 0.69]
[0.625, 0.645625, 0.665625, 0.690625, 0.67875]
[0.583125, 0.629375, 0.670625, 0.678125, 0.681875]
[0.6775, 0.6875, 0.644375, 0.6675, 0.6275]
[0.6475, 0.610625, 0.66, 0.628125, 0.58]


In [14]:
seeds = [3,4,5,6,7]
alphas = [0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
topicSentimentList = []
results = []
for alpha in alphas:
    res = []
    for seed in seeds:
        sampler = SentimentLDAGibbsSampler(10, alpha, 0.1, 0.3, seed=seed)
        sampler.run(tqdm(df['text'].tolist()), sentiFun, 100, "./lda.dll", True)
        topicSentimentList = sampler.getTopKWords(25)
        df['predicted'] = df.progress_apply(lambda row: getTweetSentiment(row.name, row['text'].split()), axis=1)
        df['predicted'] = df.progress_apply(lambda row: 4 if row['predicted'] > 0 else 0, axis=1)
        df['predict_correct'] = df.progress_apply(lambda row: row['sentiment'] == row['predicted'], axis=1)
        accuracy = df['predict_correct'].value_counts()[True] / n
        res.append(accuracy)
    results.append(res)

100%|██████████| 1600/1600 [00:00<00:00, 97621.41it/s]
